In [1]:
!sudo apt-get install -y fonts-nanum* | tail -n 1
!sudo fc-cache -fv
!rm -rf ~/.cache/matplotlib

debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 4.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Processing triggers for fontconfig (2.13.1-4.2ubuntu5) ...
/usr/share/fonts: caching, new cache contents: 0 fonts, 1 dirs
/usr/share/fonts/truetype: caching, new cache contents: 0 fonts, 3 dirs
/usr/share/fonts/truetype/humor-sans: caching, new cache contents: 1 fonts, 0 dirs
/usr/share/fonts/truetype/liberation: caching, new cache contents: 16 fonts, 0 dirs
/usr/share/fonts/truetype/nanum: caching, new cache contents: 39 fonts, 0 dirs
/usr/local/share/fonts: caching, new cache contents: 0 fonts, 0 dirs
/root/.local/share/fonts: skipping, no suc

In [1]:
# 라이브러리 임포트

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# 폰트 관련 용도
import matplotlib.font_manager as fm

# 나눔 고딕 폰트의 경로 명시
path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
font_name = fm.FontProperties(fname=path, size=10).get_name()

In [2]:
# 기본 설정값 변경

# 기본 폰트 설정
plt.rcParams['font.family'] = font_name

# 기본 폰트 사이즈 변경
# 필요에 따라 설정할 때는, plt.legend(fontsize=14)
plt.rcParams['font.size'] = 14

# 기본 그래프 사이즈 변경
# 필요에 따라 설정할 때는, plt.figure(figsize=(6,6))
plt.rcParams['figure.figsize'] = (6,6)

# 기본 그리드 표시
# 필요에 따라 설정할 때는, plt.grid()
plt.rcParams['axes.grid'] = True

# 마이너스 기호 정상 출력
plt.rcParams['axes.unicode_minus'] = False

In [3]:
import numpy as np

np.eye(20)

array([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc,
    precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize
from collections import Counter

# 재현성 설정
torch.manual_seed(42)
np.random.seed(42)

In [5]:
# 불균형 데이터 셋
class ImbalancedDataset(Dataset):
    def __init__(self, n_samples=2000, n_features=20, n_classes=4,
                 imbalance_ratio=[0.5, 0.3, 0.15, 0.05]):
        self.n_classes = n_classes
        samples_per_class = [int(n_samples * ratio) for ratio in imbalance_ratio]

        X_list = []
        y_list = []

        for class_idx in range(n_classes):
            n = samples_per_class[class_idx]
            mean = np.random.randn(n_features) * (0.5 + class_idx + 1)
            # 각 클래스 중심(평균위치) 설정 (class_idx) 커지면 원점에서 멀리 떨어지게 설정
            cov = np.eye(n_features) * (0.5 + class_idx * 0.2)
            # cov(공분산 행렬)
            X_class = np.random.multivariate_normal(mean, cov, n)
            # 설정한 mean(평균), cov(공분산) 고려, n개의 데이터 포이트(특성)에 해당하는 클래스 레이블 생성
            y_class = np.full(n, class_idx)

            X_list.append(X_class)
            y_list.append(y_class)

        self.X = torch.FloatTensor(np.vstack(X_list)) # feature matrix
        self.y = torch.LongTensor(np.hstack(y_list))  # target vector

        class_counts = Counter(self.y.numpy())
        print(f'전체 클래스 분포: {dict(sorted(class_counts.items()))}')

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [8]:
# 모델 정의
class MultiClassifier(nn.Module):
    def __init__(self, input_dim = 20, hidden_dim=64, n_classes=4):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, n_classes)
        )

    def forward(self, x):
        return self.network(x)

In [10]:
torch.ones(1) * 1.5

tensor([1.5000])

In [14]:
# 온도 스케일링
class TemperatureScaling(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)
        # 1.5 라는 값이 고정된 상수값이 아니라 역전파 (즉, 미분) 가능한 변수 설정
        # 과신을 완화시키는 역할
        # logits(모델의 예측값) >> softmax(확률 변환)
        # 확률 변환 전에 T로 나눠서 확률분포를 완화(smoothing)하는 작업을 수행하게 됨
        # 여기서 T=1 (원래 모델과 동일) / T>1 확률분포가 완만해지기 때문 >> 과신 현상 완화

    def forward(self, logits):
        return logits/self.temperature

    def calibrate(self, model, val_loader, device, max_iter=50):
        nll_criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.LBFGS([self.temperature], lr=0.01, max_iter=max_iter)

        # 검증용 데이터 전체에 대한 모델의 예측값(logits)과 정답(labels) 모으기
        logits_list = []
        labels_list = []

        model.eval()
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                logits = model(inputs)

                logits_list.append(logits)
                labels_list.append(labels)

        logits = torch.cat(logits_list) # [[logit1, ...logit100], [logit1, ...logit100], [logit1, ...logit100]]
        labels = torch.cat(labels_list)
        # 배치단위로 나누어서 계산 했으니깐 합쳐줘야 해요
        # 최종적으로 [N, 클래스 수(여기서는 4개)]

        def eval_loss():
            optimizer.zero_grad()
            loss = nll_criterion(self.forward(logits), labels)
            loss.backward()
            return loss

        optimizer.step(eval_loss)

        print(f'Temperature Scaling: T= {self.temperature.item():.3f}')
        return self.temperature.item()

In [11]:
        # 1차 미분 : gradient(경사) 순간 변화률 (특정 지점의 기울기)
        # >> 어느 방향으로 가야하나?
        # 2차 미분 : 기울기(경사) 의 변화량
        # >> (여기서부터) 얼마나 급격히 변하나? (가속도)
        # (곡률: 곡선이 얼마나 구부려져 있는가)
        # 왜 이걸 굳이 썼을까?
        # 2차 미분 하면 스칼라 값 >> 최적화해야 할 변수는 단 하나 temperature
        # LBFGS (Limited Memory Broyden Fletcher Goldfarb Shanno optimization
        # 2차 최적화 알고리즘 : 뉴턴 방법(1차 미분과 2차 미분(곡률) 동시 적용 최소값 찾는 방법)의 근사 형태 (근사치)
        # >> 이전 단계 변화량을 사용, 헤시안(Hessian) 근사치
        # >> 실제 수치해석에 들어가는 헤시안을 직접 구하지 않고 점진적으로 업데이트 나가서 근사치 구한다
        # >> 최근 10단계 업데이트된 정보만 저장해서 근사치를 구함 >> 제한된 메모리 사용
        # (일반적으로 딥러닝에 사용하는 Adam, SGD 1차 미분: gradient)

In [16]:
# 혼동행렬 시각화

def plot_confusion_matrix_with_analysis(y_true, y_pred, class_names):
    cm = confusion_matrix(y_true, y_pred)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # 혼동행렬
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[0])
    axes[0].set_title('혼동행렬 (Confusion Matrix)', fontsize=14, pad=10)
    axes[0].set_ylabel('실제 클래스', fontsize=11)
    axes[0].set_xlabel('예측 클래스', fontsize=11)

    # 오류 분석(데이터 정규화)
    # .sum(axis=1) 각 행의 합 >> 해당 클래스 중 몇 퍼센트가 틀렸는지 계산하기 위해
    # 클래스의 데이터가 불균형할 때 필수
    # + 1e-10 zero division 방지용
    cm_normalized = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-10)

    error_analysis = []
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            if i != j and cm[i, j] > 0:
                error_analysis.append({
                    'True': class_names[i],
                    'Pred': class_names[j],
                    'Count': cm[i, j],
                    'Rate': cm_normalized[i, j]
                })

    if len(error_analysis) > 0:
        error_analysis.sort(key=lambda x: x['Rate'], reverse=True)
        # x['Rate'](cm_normalized[i, j]) (오류율)기준으로 내림차순 해줘

        top_errors = error_analysis[:min(5, len(error_analysis))]
        error_labels = [f"{e['True']}→{e['Pred']}" for e in top_errors]
        error_rates = [e['Rate'] * 100 for e in top_errors]

        axes[1].barh(error_labels, error_rates, color='coral')
        axes[1].set_xlabel('오류율 (%)', fontsize=11)
        axes[1].set_title('주요 오류 유형', fontsize=14, pad=10)
        axes[1].grid(axis='x', alpha=0.3)
    else:
        axes[1].text(0.5, 0.5, '완벽한 분류!',
                    transform=axes[1].transAxes, ha='center', va='center',
                    fontsize=16)
        axes[1].axis('off')

    plt.tight_layout()
    plt.show()

    return cm

In [26]:
def calculate_detailed_metrics(y_true, y_pred, y_proba, class_names):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )

    metrics_avg = {
        'macro': precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)[:3],
        'micro': precision_recall_fscore_support(y_true, y_pred, average='micro', zero_division=0)[:3],
        'weighted': precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)[:3]
    }

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    # (1) 클래스별 지표
    x = np.arange(len(class_names))
    width = 0.25

    axes[0, 0].bar(x - width, precision, width, label='Precision', alpha=0.8)
    axes[0, 0].bar(x, recall, width, label='Recall', alpha=0.8)
    axes[0, 0].bar(x + width, f1, width, label='F1-Score', alpha=0.8)
    axes[0, 0].set_xlabel('클래스', fontsize=11)
    axes[0, 0].set_ylabel('점수', fontsize=11)
    axes[0, 0].set_title('클래스별 평가지표', fontsize=13, pad=10)
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(class_names)
    axes[0, 0].legend()
    axes[0, 0].grid(axis='y', alpha=0.3)
    axes[0, 0].set_ylim([0, 1.1])

    # (2) 평균 방식 비교
    avg_types = ['Macro', 'Micro', 'Weighted']
    avg_precisions = [metrics_avg['macro'][0], metrics_avg['micro'][0], metrics_avg['weighted'][0]]
    avg_recalls = [metrics_avg['macro'][1], metrics_avg['micro'][1], metrics_avg['weighted'][1]]
    avg_f1s = [metrics_avg['macro'][2], metrics_avg['micro'][2], metrics_avg['weighted'][2]]

    x_avg = np.arange(len(avg_types))
    axes[0, 1].bar(x_avg - width, avg_precisions, width, label='Precision', alpha=0.8)
    axes[0, 1].bar(x_avg, avg_recalls, width, label='Recall', alpha=0.8)
    axes[0, 1].bar(x_avg + width, avg_f1s, width, label='F1-Score', alpha=0.8)
    axes[0, 1].set_xlabel('평균 방식', fontsize=11)
    axes[0, 1].set_ylabel('점수', fontsize=11)
    axes[0, 1].set_title('평균 방식별 비교', fontsize=13, pad=10)
    axes[0, 1].set_xticks(x_avg)
    axes[0, 1].set_xticklabels(avg_types)
    axes[0, 1].legend()
    axes[0, 1].grid(axis='y', alpha=0.3)
    axes[0, 1].set_ylim([0, 1.1])

    # (3) Support vs F1
    axes[1, 0].scatter(support, f1, s=100, alpha=0.6, c=range(len(class_names)), cmap='viridis')
    for i, name in enumerate(class_names):
        axes[1, 0].annotate(name, (support[i], f1[i]),
                           xytext=(5, 5), textcoords='offset points', fontsize=9)
    axes[1, 0].set_xlabel('샘플 수', fontsize=11)
    axes[1, 0].set_ylabel('F1-Score', fontsize=11)
    axes[1, 0].set_title('클래스 불균형과 성능', fontsize=13, pad=10)
    axes[1, 0].grid(alpha=0.3)

    # (4) 설명
    explanation = (
        "평균 방식:\n\n"
        "• Macro: 클래스별 평균\n"
        "  모든 클래스 동등\n\n"
        "• Micro: 전체 샘플 기준\n"
        "  다수 클래스 영향\n\n"
        "• Weighted: 가중 평균\n"
        "  실제 분포 반영"
    )
    axes[1, 1].text(0.1, 0.5, explanation, transform=axes[1, 1].transAxes,
                   fontsize=11, verticalalignment='center',
                   bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))
    axes[1, 1].axis('off')

    plt.tight_layout()
    plt.show()

    print("\n" + "="*60)
    print("분류 리포트")
    print("="*60)
    print(classification_report(y_true, y_pred, target_names=class_names,
                               digits=3, zero_division=0))

    return metrics_avg

In [38]:
# ROC - AUC
def plot_roc_curves_multiclass(y_true, y_proba, class_names):
    n_classes = len(class_names)
    y_true_bin = label_binarize(y_true, classes=range(n_classes))
    # label_binarize 이진화 [0,1,2,3] >>> one-hot encoding (1,0)

    fpr = dict() # {}
    tpr = dict()
    roc_auc = dict()

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # micro 평균 및 macro 평균 계산
    # micro 평균 : 샘플 단위 합산
    fpr["micro"], tpr["micro"], _ = roc_curve(y_true_bin.ravel(), y_proba.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
        # np.interp(보간법) interpolation
    mean_tpr /= n_classes
    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    # 클래스별 ROC
    colors = plt.cm.Set3(np.linspace(0, 1, n_classes))
    for i, color in enumerate(colors):
        axes[0].plot(fpr[i], tpr[i], color=color, lw=2,
                    label=f'{class_names[i]} (AUC={roc_auc[i]:.3f})')

    axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
    axes[0].set_xlim([0.0, 1.0])
    axes[0].set_ylim([0.0, 1.05])
    axes[0].set_xlabel('False Positive Rate', fontsize=11)
    axes[0].set_ylabel('True Positive Rate', fontsize=11)
    axes[0].set_title('클래스별 ROC (One-vs-Rest)', fontsize=13, pad=10)
    axes[0].legend(loc="lower right", fontsize=9)
    axes[0].grid(alpha=0.3)

    # Macro/Micro
    axes[1].plot(fpr["micro"], tpr["micro"],
                label=f'Micro-avg (AUC={roc_auc["micro"]:.3f})',
                color='deeppink', linestyle=':', linewidth=3)
    axes[1].plot(fpr["macro"], tpr["macro"],
                label=f'Macro-avg (AUC={roc_auc["macro"]:.3f})',
                color='navy', linestyle=':', linewidth=3)
    axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')

    axes[1].set_xlim([0.0, 1.0])
    axes[1].set_ylim([0.0, 1.05])
    axes[1].set_xlabel('False Positive Rate', fontsize=11)
    axes[1].set_ylabel('True Positive Rate', fontsize=11)
    axes[1].set_title('평균 ROC', fontsize=13, pad=10)
    axes[1].legend(loc="lower right", fontsize=10)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("\n" + "="*60)
    print("ROC-AUC 점수")
    print("="*60)
    for i, name in enumerate(class_names):
        print(f"{name:15s}: {roc_auc[i]:.4f}")
    print(f"{'Micro-avg':15s}: {roc_auc['micro']:.4f}")
    print(f"{'Macro-avg':15s}: {roc_auc['macro']:.4f}")

    return roc_auc

In [39]:
def get_labels_from_loader(loader):
    """DataLoader에서 모든 레이블 추출"""
    labels = []
    for _, batch_labels in loader:
        labels.extend(batch_labels.numpy().tolist())
    return labels

In [40]:
def get_class_weights(train_loader, n_classes):
    """클래스 가중치 계산"""
    labels = get_labels_from_loader(train_loader)
    class_counts = Counter(labels)
    n_samples = len(labels)

    weights = torch.FloatTensor([
        n_samples / (n_classes * class_counts.get(i, 1))
        for i in range(n_classes)
    ])
    # weight = 전체 샘플수(n_samples) / 총 클래스 수(n_classes) * (해당 클래스의 샘플 수)
    # class_counts.get(i, 1) : get( , 1) zero division 방지 위해 기본값 1 반환

    print(f"\n학습 데이터 클래스 분포: {dict(sorted(class_counts.items()))}")
    print(f"클래스 가중치: {weights.numpy()}")
    return weights

In [43]:
def create_weighted_sampler(train_loader):
    """Oversampling을 위한 Sampler"""
    labels = get_labels_from_loader(train_loader)
    class_counts = Counter(labels)

    sample_weights = [1.0 / class_counts[label] for label in labels]
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
        # 중복 추출 허용(통계에서는 복원 추출)
        # 소수 클래스를 다수 클래스 개수 맞추기 위해 중복 추출해서 개수를 맞추겠다 (오버샘플링)
    )

    return sampler

In [44]:
class AugmentedWrapper(Dataset):
    """데이터 증강 Wrapper"""
    def __init__(self, dataset, augment_ratio=0.5, noise_std=0.15):
        self.dataset = dataset
        self.base_len = len(dataset)
        self.augment_len = int(self.base_len * augment_ratio)
        self.total_len = self.base_len + self.augment_len
        self.noise_std = noise_std

    def __len__(self):
        return self.total_len

    def __getitem__(self, idx):
        if idx < self.base_len:
            return self.dataset[idx]
        # 원본 데이터에 있는 인덱스 번호 (idx)
        else:
            # 증강 샘플
            # base_len이 100이라 가정하면 idx = [0, 99]
            # idx = 105 >> (105-100) % 100 = 5
            base_idx = (idx - self.base_len) % self.base_len
            x, y = self.dataset[base_idx]
            # dataset 에서 dataset[5] 5번 데이터를 기준(base)로 할게
            # 여기서 x(데이터), y(라벨 label)
            noise = torch.randn_like(x) * self.noise_std
            # 가우시안 노이즈 적용
            # randn_like(x) x와 똑같은 모양(shape) 행렬을 생성(평균 0, 표준편차 1)

            return x + noise, y
            # x + noise
            # >> 5번 데이터를 베이스로 하여서 노이즈를 섞은 데이터(정규분포 무작위 값_아주 작아)

            # 그러면 데이터 값이 x=100 이면?
            # 100 + 정규분포를 띈 아주 작은 랜덤 값(noise_std=0.15, 평균0, 표준편차 1)
            # 왜냐하면 N(mean, sigma^2) = (100, (0.15)^2)

In [47]:
(0.15)**2

0.0225

In [25]:
# 참고

import numpy as np

ex = np.array([[50, 10],
               [5, 36]])

print(ex)
print()

# 정규화
# 각 행(row) 합으로 나누기
ex_normalized = ex.astype('float') / (ex.sum(axis=1)[:, np.newaxis] + 1e-10)
print(ex.astype('float'))
print()
print(ex.sum(axis=1))
print()
print(ex.sum(axis=1)[:, np.newaxis])
print()
print(ex_normalized)

[[50 10]
 [ 5 36]]

[[50. 10.]
 [ 5. 36.]]

[60 41]

[[60]
 [41]]

[[0.83333333 0.16666667]
 [0.12195122 0.87804878]]


In [27]:
ex

array([[50, 10],
       [ 5, 36]])

In [30]:
ex.shape

(2, 2)

In [28]:
ex.ravel()

array([50, 10,  5, 36])

In [29]:
np.zeros_like(ex)

array([[0, 0],
       [0, 0]])

In [37]:
n_classes = 3

# 클래스 별 fpr
fpr = {
    0 : np.array([0.0, 0.1, 0.2, 0.3]),
    1 : np.array([0.0, 0.2, 0.4, 0.6]),
    2 : np.array([0.0, 0.1, 0.3, 0.5])
}

# 클래스 별 tpr
tpr = {
    0 : np.array([0.0, 0.6, 0.8, 1.0]),
    1 : np.array([0.0, 0.4, 0.8, 1.0]),
    2 : np.array([0.0, 0.3, 0.6, 0.9])
}

# 모든 fpr 값 합치기
# 중복 제거 뒤 합치기 (unique)
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
# print(all_fpr)
# [0.  0.1 0.2 0.3 0.4 0.5 0.6]

# 평균 trp 저장할 배열 생성
mean_tpr = np.zeros_like(all_fpr)
# print(mean_tpr)
# [0. 0. 0. 0. 0. 0. 0.]

for i in range(n_classes):
    interpolated_tpr = np.interp(all_fpr, fpr[i], tpr[i])
    # print(interpolated_tpr)

    # 평균 계산 위해 누적
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])

print(mean_tpr)

print(mean_tpr / n_classes)
# [0.         0.36666667 0.55       0.73333333 0.85       0.93333333    0.96666667]
# 이게 뭐요?
# mean_tpr라고 하는디, 여러 클래스 roc curve(fpr - tpr)를 평균낸 roc curve 요(이게 대표 roc curve랑께)

# 왜 이렇게 해요?

# 클래스마다 fpr 좌표가 다르네....
# 서로 좌표가 달라서 평균 불가능
# 공통으로 가지고 있는 동일한 x축(all_fpr) 기준, tpr 값 맞춰서 계산한 뒤, 평균 계산해


[0.   1.1  1.65 2.2  2.55 2.8  2.9 ]
[0.         0.36666667 0.55       0.73333333 0.85       0.93333333
 0.96666667]
